In [20]:
# ============================================================
# BAYESIAN NETWORK FOR HEART DISEASE DIAGNOSIS
# Cleveland Heart Disease Dataset
# ============================================================

import pandas as pd
import numpy as np

from itertools import product

print("=" * 70)
print("BAYESIAN NETWORK - HEART DISEASE DIAGNOSIS")
print("=" * 70)





# ============================================================
# 2. LOAD DATASET
# ============================================================

columns = [
    'age',
    'sex',
    'cp',
    'trestbps',
    'chol',
    'fbs',
    'restecg',
    'thalach',
    'exang',
    'oldpeak',
    'slope',
    'ca',
    'thal',
    'target'
]

data = pd.read_csv(r"D:\Downloads\abhishek (1)\abhishek\BI-codes\Heart_disease_cleveland_new.csv")
    


print("\nInitial dataset shape:")
print(data.shape)

print("\nFirst 5 records:")
print(data.head())



BAYESIAN NETWORK - HEART DISEASE DIAGNOSIS

Initial dataset shape:
(303, 14)

First 5 records:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   0       145   233    1        2      150      0      2.3      2   
1   67    1   3       160   286    0        2      108      1      1.5      1   
2   67    1   3       120   229    0        2      129      1      2.6      1   
3   37    1   2       130   250    0        0      187      0      3.5      2   
4   41    0   1       130   204    0        2      172      0      1.4      0   

   ca  thal  target  
0   0     2       0  
1   3     1       1  
2   2     3       1  
3   0     1       0  
4   0     1       0  


In [21]:
# ============================================================
# 3. CLEAN DATA
# ============================================================

# Convert '?' to NaN
data = data.replace('?', np.nan)

# Convert all columns to numeric
for column in columns:
    data[column] = pd.to_numeric(
        data[column],
        errors='coerce'
    )

print("\nMissing values:")
print(data.isnull().sum())

# Remove rows containing missing values
data = data.dropna().reset_index(drop=True)

print("\nDataset shape after removing missing values:")
print(data.shape)



Missing values:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

Dataset shape after removing missing values:
(303, 14)


In [22]:
# ============================================================
# 4. CONVERT TARGET VARIABLE
# ============================================================

# Cleveland dataset:
# num = 0  -> No heart disease
# num = 1-4 -> Heart disease

data['HeartDisease'] = np.where(
    data['target'] == 0,
    0,
    1
)

data.drop(
    columns=['target'],
    inplace=True
)


# ============================================================
# 5. DISCRETIZE CONTINUOUS VARIABLES
# ============================================================

# Age
data['age'] = pd.cut(
    data['age'],
    bins=[-np.inf, 40, 60, np.inf],
    labels=[0, 1, 2]
).astype(int)

# Resting blood pressure
data['trestbps'] = pd.cut(
    data['trestbps'],
    bins=[-np.inf, 120, 140, np.inf],
    labels=[0, 1, 2]
).astype(int)

# Cholesterol
data['chol'] = pd.cut(
    data['chol'],
    bins=[-np.inf, 200, np.inf],
    labels=[0, 1]
).astype(int)

# Maximum heart rate
data['thalach'] = pd.cut(
    data['thalach'],
    bins=[-np.inf, 120, np.inf],
    labels=[0, 1]
).astype(int)

# ST depression
data['oldpeak'] = pd.cut(
    data['oldpeak'],
    bins=[-np.inf, 1, np.inf],
    labels=[0, 1]
).astype(int)


print("\n" + "=" * 70)
print("CLEANED AND DISCRETIZED DATA")
print("=" * 70)

print(data.head())

print("\nFinal shape:")
print(data.shape)




CLEANED AND DISCRETIZED DATA
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0    2    1   0         2     1    1        2        1      0        1      2   
1    2    1   3         2     1    0        2        0      1        1      1   
2    2    1   3         0     1    0        2        1      1        1      1   
3    0    1   2         1     1    0        0        1      0        1      2   
4    1    0   1         1     1    0        2        1      0        1      0   

   ca  thal  HeartDisease  
0   0     2             0  
1   3     1             1  
2   2     3             1  
3   0     1             0  
4   0     1             0  

Final shape:
(303, 14)


In [ ]:
# ============================================================
# 6. BAYESIAN NETWORK STRUCTURE
# ============================================================

parents = [
    'age',
    'sex',
    'cp',
    'trestbps',
    'chol',
    'fbs',
    'restecg',
    'thalach',
    'exang',
    'oldpeak',
    'slope',
    'ca',
    'thal'
]

target = 'HeartDisease'


print("\n" + "=" * 70)
print("BAYESIAN NETWORK STRUCTURE")
print("=" * 70)

print("\nEdges:")

for parent in parents:
    print(
        f"{parent} --> {target}"
    )


# ============================================================
# 7. GET STATES
# ============================================================

states = {}

for column in data.columns:
    states[column] = sorted(
        data[column].unique().tolist()
    )

print("\nVariable states:")

for column in states:
    print(
        f"{column}: {states[column]}"
    )


# ============================================================
# 8. CALCULATE PRIOR PROBABILITIES
# ============================================================

target_counts = data[target].value_counts()

total_records = len(data)

prior_probability = {}

for state in states[target]:

    prior_probability[state] = (
        target_counts.get(state, 0)
        / total_records
    )


print("\n" + "=" * 70)
print("PRIOR PROBABILITIES")
print("=" * 70)

for state, probability in prior_probability.items():

    if state == 0:
        label = "No Heart Disease"
    else:
        label = "Heart Disease"

    print(
        f"{label}: {probability:.4f}"
    )


# ============================================================
# 9. BAYESIAN NETWORK CONDITIONAL PROBABILITY
# ============================================================

def calculate_probability(
    patient,
    target_state
):

    # Start with prior probability
    probability = prior_probability[
        target_state
    ]

    target_data = data[
        data[target] == target_state
    ]

    # Naive Bayesian conditional probability
    # P(H | X1,...,Xn) proportional to
    # P(H) * P(X1|H) * ... * P(Xn|H)

    for feature in parents:

        feature_value = patient[feature]

        count_feature_and_target = len(
            target_data[
                target_data[feature]
                == feature_value
            ]
        )

        count_target = len(target_data)

        # Laplace smoothing
        number_of_states = len(
            states[feature]
        )

        conditional_probability = (
            count_feature_and_target + 1
        ) / (
            count_target
            + number_of_states
        )

        probability *= conditional_probability

    return probability


# ============================================================
# 10. NORMALIZE PROBABILITIES
# ============================================================

def diagnose_patient(patient:dict):

    probability_no_disease = (
        calculate_probability(
            patient,
            0
        )
    )

    probability_disease = (
        calculate_probability(
            patient,
            1
        )
    )

    total = (
        probability_no_disease
        + probability_disease
    )

    probability_no_disease /= total
    probability_disease /= total

    if probability_disease > probability_no_disease:

        prediction = "HEART DISEASE"

    else:

        prediction = "NO HEART DISEASE"

    return (
        probability_no_disease,
        probability_disease,
        prediction
    )



BAYESIAN NETWORK STRUCTURE

Edges:
age --> HeartDisease
sex --> HeartDisease
cp --> HeartDisease
trestbps --> HeartDisease
chol --> HeartDisease
fbs --> HeartDisease
restecg --> HeartDisease
thalach --> HeartDisease
exang --> HeartDisease
oldpeak --> HeartDisease
slope --> HeartDisease
ca --> HeartDisease
thal --> HeartDisease

Variable states:
age: [0, 1, 2]
sex: [0, 1]
cp: [0, 1, 2, 3]
trestbps: [0, 1, 2]
chol: [0, 1]
fbs: [0, 1]
restecg: [0, 1, 2]
thalach: [0, 1]
exang: [0, 1]
oldpeak: [0, 1]
slope: [0, 1, 2]
ca: [0, 1, 2, 3]
thal: [1, 2, 3]
HeartDisease: [0, 1]

PRIOR PROBABILITIES
No Heart Disease: 0.5413
Heart Disease: 0.4587


In [24]:
# ============================================================
# 11. PATIENT 1
# ============================================================

patient_1 = {
    'age': 2,
    'sex': 1,
    'cp': 4,
    'trestbps': 2,
    'chol': 1,
    'fbs': 0,
    'restecg': 2,
    'thalach': 0,
    'exang': 1,
    'oldpeak': 1,
    'slope': 2,
    'ca': 1,
    'thal': 3
}


print("\n" + "=" * 70)
print("PATIENT 1 DIAGNOSIS")
print("=" * 70)

print("\nPatient data:")

for feature, value in patient_1.items():
    print(
        f"{feature}: {value}"
    )


p_no_1, p_yes_1, prediction_1 = (
    diagnose_patient(patient_1)
)

print("\nProbability of NO heart disease:")
print(
    f"{p_no_1:.4f}"
)

print("\nProbability of HEART disease:")
print(
    f"{p_yes_1:.4f}"
)

print("\nPrediction:")
print(prediction_1)


# ============================================================
# 12. PATIENT 2
# ============================================================

patient_2 = {
    'age': 1,
    'sex': 0,
    'cp': 2,
    'trestbps': 1,
    'chol': 0,
    'fbs': 0,
    'restecg': 0,
    'thalach': 1,
    'exang': 0,
    'oldpeak': 0,
    'slope': 1,
    'ca': 0,
    'thal': 3
}


print("\n" + "=" * 70)
print("PATIENT 2 DIAGNOSIS")
print("=" * 70)

print("\nPatient data:")

for feature, value in patient_2.items():
    print(
        f"{feature}: {value}"
    )


p_no_2, p_yes_2, prediction_2 = (
    diagnose_patient(patient_2)
)

print("\nProbability of NO heart disease:")
print(
    f"{p_no_2:.4f}"
)

print("\nProbability of HEART disease:")
print(
    f"{p_yes_2:.4f}"
)

print("\nPrediction:")
print(prediction_2)


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL DIAGNOSIS SUMMARY")
print("=" * 70)

print(
    f"\nPatient 1: {prediction_1}"
)

print(
    f"Patient 1 - P(No Disease) = {p_no_1:.4f}"
)

print(
    f"Patient 1 - P(Disease) = {p_yes_1:.4f}"
)

print(
    f"\nPatient 2: {prediction_2}"
)

print(
    f"Patient 2 - P(No Disease) = {p_no_2:.4f}"
)

print(
    f"Patient 2 - P(Disease) = {p_yes_2:.4f}"
)

print("\n" + "=" * 70)
print("BAYESIAN NETWORK DIAGNOSIS COMPLETED")
print("=" * 70)




PATIENT 1 DIAGNOSIS

Patient data:
age: 2
sex: 1
cp: 4
trestbps: 2
chol: 1
fbs: 0
restecg: 2
thalach: 0
exang: 1
oldpeak: 1
slope: 2
ca: 1
thal: 3

Probability of NO heart disease:
0.0004

Probability of HEART disease:
0.9996

Prediction:
HEART DISEASE

PATIENT 2 DIAGNOSIS

Patient data:
age: 1
sex: 0
cp: 2
trestbps: 1
chol: 0
fbs: 0
restecg: 0
thalach: 1
exang: 0
oldpeak: 0
slope: 1
ca: 0
thal: 3

Probability of NO heart disease:
0.9554

Probability of HEART disease:
0.0446

Prediction:
NO HEART DISEASE

FINAL DIAGNOSIS SUMMARY

Patient 1: HEART DISEASE
Patient 1 - P(No Disease) = 0.0004
Patient 1 - P(Disease) = 0.9996

Patient 2: NO HEART DISEASE
Patient 2 - P(No Disease) = 0.9554
Patient 2 - P(Disease) = 0.0446

BAYESIAN NETWORK DIAGNOSIS COMPLETED


In [ ]:
# 🫀 Bayesian Network – Heart Disease Diagnosis (मराठीत)

## भाग 1: Basic Terms

| Term | सोप्या भाषेत अर्थ |
|---|---|
| **Probability (शक्यता)** | एखादी गोष्ट घडण्याची शक्यता, 0 ते 1 मध्ये. उदा. 0.54 = 54% |
| **Prior Probability P(H)** | कोणतीही माहिती माहीत नसताना असलेली शक्यता. इथे: रुग्णांपैकी किती % ना heart disease आहे |
| **Conditional Probability P(A\|B)** | B झालं असं माहीत असताना A होण्याची शक्यता |
| **Bayes' Theorem** | P(H\|X) = P(X\|H) × P(H) / P(X). नवीन माहिती (evidence) मिळाल्यावर शक्यता update करण्याचं सूत्र |
| **Posterior Probability** | Evidence पाहिल्यानंतरची अंतिम शक्यता, म्हणजे P(H\|X) |
| **Bayesian Network** | Graph जिथे प्रत्येक node = variable आणि प्रत्येक arrow = त्यांच्यातील संबंध (dependency). Probability आणि graph यांचं मिश्रण |
| **Naive Bayes** | असं गृहीत धरतो की सर्व features एकमेकांपासून independent आहेत (class दिलेला असताना). म्हणून "naive" |
| **Discretization** | Continuous value (उदा. age 54) → category (0/1/2) मध्ये बदलणे |
| **Laplace Smoothing** | Count मध्ये +1 करणे, म्हणजे कधीही probability = 0 येऊ नये |
| **Normalization** | दोन्ही probabilities चा बेरीज 1 व्हावा म्हणून sum ने भागणे |
| **Feature / Target** | Feature = इनपुट (age, chol...). Target = जे ओळखायचं ते (HeartDisease) |
| **NaN** | Missing value |

**Dataset Columns (Cleveland):**
`age` वय · `sex` लिंग (1=पुरुष) · `cp` छातीतील दुखण्याचा प्रकार · `trestbps` विश्रांतीतील BP · `chol` cholesterol · `fbs` fasting blood sugar · `restecg` ECG result · `thalach` max heart rate · `exang` exercise मुळे angina · `oldpeak` ST depression · `slope` ST segment slope · `ca` रंगीत झालेल्या मुख्य vessels ची संख्या · `thal` thalassemia · `num` diagnosis (0 = नाही, 1-4 = आहे)

---

## भाग 2: Line-by-Line Explanation

### Cell 1: Imports आणि Data Load

```python
import pandas as pd
import numpy as np
from google.colab import files
from itertools import product
```
- `pandas`: table (DataFrame) स्वरूपात data हाताळण्यासाठी.
- `numpy`: गणित, `NaN`, `inf` वापरण्यासाठी.
- `google.colab.files`: Colab मध्ये आपल्या computer वरून file upload करण्यासाठी.
- `product`: import केलंय पण **वापरलेलं नाही** (viva मध्ये विचारू शकतात).

```python
print("=" * 70)
```
70 वेळा `=` छापतो, फक्त सुंदर heading दिसण्यासाठी.

```python
uploaded = files.upload()
filename = list(uploaded.keys())[0]
```
- `files.upload()` upload box उघडतो. Upload झालेल्या files चा dictionary मिळतो `{filename: content}`.
- `.keys()[0]` ने पहिल्या file चं नाव घेतो, पुढे वापरण्यासाठी.

```python
columns = ['age','sex',...,'num']
data = pd.read_csv(filename, header=None, names=columns)
```
- Cleveland CSV मध्ये **header (column names) नसतात**, म्हणून `header=None`.
- `names=columns` ने आपण स्वतः column names देतो.
- `data.shape` = (rows, columns), `data.head()` = पहिले 5 rows.

### Cell 2: Data Cleaning

```python
data = data.replace('?', np.nan)
```
Dataset मध्ये missing values `?` ने दाखवलेले असतात. ते `NaN` (proper missing value) मध्ये बदलतो.

```python
for column in columns:
    data[column] = pd.to_numeric(data[column], errors='coerce')
```
`?` मुळे columns text (object) झालेले असतात. त्यांना number मध्ये बदलतो. `errors='coerce'` म्हणजे convert न झालेलं काहीही `NaN` होईल.

```python
print(data.isnull().sum())
data = data.dropna().reset_index(drop=True)
```
- प्रत्येक column मध्ये किती missing values आहेत ते दाखवतो (सहसा `ca` मध्ये 4 आणि `thal` मध्ये 2).
- `dropna()` ने missing असलेले rows काढतो. त्यामुळे साधारण 303 → 297 rows उरतात.
- `reset_index(drop=True)` ने index पुन्हा 0 पासून लावतो.

### Cell 3: Target Convert आणि Discretize

```python
data['HeartDisease'] = np.where(data['num'] == 0, 0, 1)
data.drop(columns=['num'], inplace=True)
```
- `num` मध्ये 0 ते 4 values आहेत. आपल्याला फक्त **दोन class** हवेत (binary): 0 = रोग नाही, 1 = रोग आहे (1,2,3,4 सगळे).
- `np.where(condition, a, b)`: condition खरी तर `a`, नाहीतर `b`.
- जुना `num` column काढून टाकतो.

**Discretization — का?** आपला algorithm count वर आधारित आहे (किती रुग्णांचं age = 54 होतं?). Continuous values खूप वेगवेगळ्या असतात, म्हणून त्यांना गटांत (bins) विभागतो.

```python
data['age'] = pd.cut(data['age'], bins=[-np.inf, 40, 60, np.inf], labels=[0,1,2]).astype(int)
```
- `pd.cut` values ना bins मध्ये टाकतो.
- Age ≤ 40 → **0** (तरुण), 41–60 → **1** (मध्यम), > 60 → **2** (वृद्ध).
- `.astype(int)`: `pd.cut` category type देतो, त्याला पुन्हा integer करतो.

तसंच बाकी:

| Column | 0 | 1 | 2 |
|---|---|---|---|
| trestbps (BP) | ≤120 सामान्य | 121–140 | >140 जास्त |
| chol | ≤200 सामान्य | >200 जास्त | — |
| thalach | ≤120 कमी | >120 | — |
| oldpeak | ≤1 | >1 | — |

### Cell 4: Network Structure, Prior आणि Probability

```python
parents = ['age','sex',...,'thal']
target = 'HeartDisease'
for parent in parents:
    print(f"{parent} --> {target}")
```
Network चे edges छापतो, म्हणजे प्रत्येक feature पासून HeartDisease कडे arrow. एकूण 13 edges.

```python
states = {}
for column in data.columns:
    states[column] = sorted(data[column].unique().tolist())
```
प्रत्येक column मध्ये कोणत्या वेगळ्या values (states) आहेत त्याचा dictionary. उदा. `sex: [0,1]`. हे **Laplace smoothing** साठी लागतं (किती states आहेत ते मोजायला).

```python
target_counts = data[target].value_counts()
total_records = len(data)
prior_probability[state] = target_counts.get(state, 0) / total_records
```
**Prior** = (त्या class चे रुग्ण) ÷ (एकूण रुग्ण). उदा. 137/297 = 0.46 रोग असलेले.

**Main function — `calculate_probability(patient, target_state)`:**

```python
probability = prior_probability[target_state]
```
सुरुवात prior ने: P(H).

```python
target_data = data[data[target] == target_state]
```
फक्त त्या class चे रुग्ण वेगळे काढतो (उदा. फक्त रोग असलेले).

```python
for feature in parents:
    feature_value = patient[feature]
    count_feature_and_target = len(target_data[target_data[feature] == feature_value])
    count_target = len(target_data)
```
प्रत्येक feature साठी: त्या class च्या रुग्णांपैकी किती जणांची ही feature value आपल्या patient सारखीच आहे ते मोजतो.

```python
number_of_states = len(states[feature])
conditional_probability = (count_feature_and_target + 1) / (count_target + number_of_states)
probability *= conditional_probability
```
- **Laplace smoothing**: `(count + 1) / (total + states)`. Count 0 असला तरी probability 0 होत नाही, नाहीतर संपूर्ण गुणाकार 0 होईल.
- सूत्र: **P(H|X) ∝ P(H) × P(X₁|H) × P(X₂|H) × … × P(Xₙ|H)**
- सगळे P(Xᵢ|H) एकमेकांत गुणत जातो.

**`diagnose_patient(patient)`:**

```python
p_no = calculate_probability(patient, 0)
p_yes = calculate_probability(patient, 1)
total = p_no + p_yes
p_no /= total
p_yes /= total
```
- दोन्ही classes साठी probability काढतो. ही raw (अपूर्ण) values असतात, म्हणून त्यांची बेरीज = 1 होईल असं **normalize** करतो.
- जो मोठा तोच prediction: `if p_yes > p_no → "HEART DISEASE"`.
- Function तीन गोष्टी return करतो: p_no, p_yes, prediction.

### Cell 5: Patients ची Testing

`patient_1` आणि `patient_2` हे dictionaries आहेत, ज्यात प्रत्येक feature ची value दिलेली आहे (discretized/coded स्वरूपात). `diagnose_patient()` ला पाठवून probabilities आणि prediction मिळवतो. शेवटी **Final Summary** मध्ये दोन्ही patients चे निकाल एकत्र छापतो. शेवटची note: हे शैक्षणिक demo आहे, खरं medical diagnosis नाही.

---

## भाग 3: Viva Q&A

**Q1. Bayesian Network म्हणजे काय?**
उत्तर: Directed Acyclic Graph (DAG) ज्यात nodes = random variables, edges = त्यांच्यातील probabilistic dependency आणि प्रत्येक node सोबत Conditional Probability Table (CPT) असते.

**Q2. Bayes' theorem सांगा.**
उत्तर: P(H|X) = [P(X|H) × P(H)] / P(X)

**Q3. Prior आणि Posterior probability मध्ये फरक?**
उत्तर: Prior = evidence आधीची शक्यता. Posterior = evidence पाहिल्यानंतरची updated शक्यता.

**Q4. या program मध्ये प्रत्यक्षात कोणता algorithm वापरलाय?**
उत्तर: **Naive Bayes** पद्धत. कारण सर्व features ला independent मानून probabilities गुणलेल्या आहेत. Network चं structure Naive Bayes सारखंच (एक target आणि 13 features) आहे.

**Q5. Laplace smoothing का वापरली?**
उत्तर: एखादी feature value एखाद्या class मध्ये training data मध्ये आली नसेल तर count = 0 आणि probability = 0 होईल. गुणाकारामुळे संपूर्ण result 0 होईल. +1 केल्याने हे टाळता येतं.

**Q6. Normalization का केली?**
उत्तर: Raw values खूप छोट्या असतात आणि त्यांची बेरीज 1 नसते. Sum ने भागल्यावर त्या proper probabilities (बेरीज = 1) होतात.

**Q7. Discretization का केली?**
उत्तर: कारण हा counting-based (tabular CPT) approach आहे. Continuous values ना count करता येत नाही, म्हणून त्यांना categories मध्ये बदलावं लागतं.

**Q8. `num` column चं काय केलं?**
उत्तर: 0 = रोग नाही, 1 ते 4 = रोग आहे. त्याला binary target `HeartDisease` (0/1) मध्ये बदललं.

**Q9. Missing values चं काय केलं?**
उत्तर: `?` ला NaN मध्ये बदललं आणि `dropna()` ने त्या rows काढल्या.

**Q10. Naive assumption चा तोटा काय?**
उत्तर: प्रत्यक्षात features independent नसतात (उदा. age आणि BP यांचा संबंध असतो). तरीही Naive Bayes चांगलं काम करतो.

**Q11. Conditional Probability Table (CPT) म्हणजे काय?**
उत्तर: प्रत्येक node ची तिच्या parents च्या सर्व combinations साठीची probability ची table.

**Q12. `pd.cut` आणि `np.where` काय करतात?**
उत्तर: `pd.cut` = continuous values ना bins मध्ये विभागतो. `np.where` = condition वरून दोन values मधून एक निवडतो.

**Q13. `header=None` का दिलं?**
उत्तर: CSV मध्ये column names नाहीत, म्हणून पहिली ओळ डेटा समजली पाहिजे.

**Q14. Accuracy कशी तपासाल?**
उत्तर: Data ला train/test मध्ये विभागून test set वर prediction करून accuracy, precision, recall, confusion matrix काढता येईल. या code मध्ये ते केलेलं नाही, फक्त 2 patients वर demo आहे.

**Q15. जर features जास्त असतील तर काय problem येईल?**
उत्तर: खूप छोट्या probabilities गुणल्याने **underflow** (value 0 च्या जवळ) होऊ शकतो. उपाय: **log probabilities** वापरून बेरीज करणे.

**Q16. Bayesian Network आणि Naive Bayes मध्ये फरक?**
उत्तर: Naive Bayes हा Bayesian Network चा साधा प्रकार आहे, ज्यात एकच parent (class) आणि features मध्ये आपापसात edges नसतात. सामान्य Bayesian Network मध्ये features मध्येही dependencies दाखवता येतात.

---

## ⚠️ Viva साठी एक महत्त्वाची टीप

तुमच्या code मध्ये edges **feature → HeartDisease** असे छापले आहेत, पण गणित (formula) **Naive Bayes** चं आहे, ज्यात खरं तर arrows **HeartDisease → features** असतात. Examiner ने विचारलं तर सांगा: *"Structure दाखवण्यासाठी edges छापले आहेत, पण probability calculation Naive Bayes assumption वर आधारित आहे."* हे सांगितलंत तर तुम्ही नीट समजून घेतलंय हे दिसेल.

तसंच `product` import वापरलेला नाही, हेही लक्षात ठेवा.

All the best for the practical! 🙂 काही भाग अजून सोपा हवा असेल तर सांग.